In [1]:
import pandas as pd
import numpy as np
import random
import requests
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error
from rouge_score import rouge_scorer
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'sentence_transformers'

In [2]:
# ===========================
# 1. Load data
# ===========================
df = pd.read_csv("../outputs/unified_behavior_with_archetype.csv")
print(f"Total reviews: {len(df)}")

# Split by user (80/20)
train_list, test_list = [], []
for user, group in df.groupby('user_id'):
    if len(group) >= 2:
        train, test = train_test_split(group, test_size=0.2, random_state=42)
        train_list.append(train)
        test_list.append(test)
    else:
        train_list.append(group)
train_df = pd.concat(train_list)
test_df = pd.concat(test_list)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")


Total reviews: 28294
Train: 25135, Test: 3159


In [3]:
# ===========================
# 2. Helper functions (copied/adapted from task_a.py)
# ===========================
def get_few_shot_examples(archetype, target_restaurant, unified_df, n=3):
    """Sample past reviews from same archetype."""
    archetype_reviews = unified_df[unified_df["archetype"] == archetype]
    if len(archetype_reviews) == 0:
        archetype_reviews = unified_df
    sample_size = min(n, len(archetype_reviews))
    sampled = archetype_reviews.sample(sample_size, random_state=42)
    examples = []
    for _, row in sampled.iterrows():
        examples.append({
            "rating": row["rating"],
            "review_text": row["review_text"][:200]
        })
    return examples

def build_prompt(persona, target_restaurant, context, few_shots):
    archetype = persona.get("archetype", "Balanced")
    dominant_value = persona.get("dominant_value", "neutral")
    avg_rating = persona.get("avg_rating", 3.5)
    
    context_tones = {
        "celebration": "Excited, generous",
        "sapa_budget": "Price‑sensitive, critical",
        "general": "Balanced and honest"
    }
    tone = context_tones.get(context, "Balanced and honest")
    
    prompt = f"""You are simulating a Nigerian user on a restaurant review platform.

**User Persona:**
- Archetype: {archetype}
- Dominant value: {dominant_value}
- Typical rating: {avg_rating:.1f} stars
- Current context: {context} → {tone}

**Few-shot examples of this user's past reviews:**
"""
    for i, ex in enumerate(few_shots, 1):
        prompt += f"{i}. Rating: {ex['rating']} stars\n   Review: {ex['review_text']}\n"
    
    prompt += f"""
**Now write a NEW review for this restaurant:**
Name: {target_restaurant.get('name', 'Unknown')}
Category: {target_restaurant.get('category', 'Nigerian')}
Price range: {target_restaurant.get('price_range', 'moderate')}
Location: {target_restaurant.get('location_type', 'Lagos')}
Description: {target_restaurant.get('description', 'A local spot')}

**Output exactly in this format:**
Rating: (1-5 integer)
Review: (natural language review in the user's voice, 50-150 words. Write in a natural mix of standard English and occasional Nigerian Pidgin/expressions – code‑switching.)

Do not add any extra text.
"""
    return prompt

def call_llm(prompt, api_key):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 300
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"LLM error {response.status_code}: {response.text}")
        return None

def parse_response(response):
    lines = response.strip().split("\n")
    rating = 3
    review = "Could not parse."
    for line in lines:
        if line.lower().startswith("rating:"):
            try:
                rating = int(line.split(":")[1].strip())
                rating = max(1, min(5, rating))
            except:
                rating = 3
        elif line.lower().startswith("review:"):
            review = line.split(":", 1)[1].strip()
    return rating, review

In [6]:
# ===========================
# 3. Evaluation loop (small subset for speed)
# ===========================

import time

API_KEY = os.getenv("GROQ_API_KEY")  # or set directly
if not API_KEY:
    raise ValueError("Set GROQ_API_KEY environment variable")

sample_size = 30  # adjust based budget
eval_df = test_df.sample(min(sample_size, len(test_df)), random_state=42)

pred_ratings = []
true_ratings = []
generated_reviews = []
true_reviews = []

for i, (idx, row) in enumerate(eval_df.iterrows()):
    user_id = row['user_id']
    true_rating = row['rating']
    true_review = row['review_text']
    
    # Get persona from train_df (fallback to random if no training review for user)
    user_train = train_df[train_df['user_id'] == user_id]
    if len(user_train) > 0:
        persona = user_train.iloc[0]
    else:
        persona = train_df.sample(1, random_state=42).iloc[0]
    
    # Build target restaurant dict (simplified)
    target = {
        'name': row.get('item_id', 'Unknown'),
        'category': 'Restaurant',
        'price_range': 'Moderate',
        'location_type': 'Lagos',
        'description': row['review_text'][:200]
    }
    
    few_shots = get_few_shot_examples(persona['archetype'], target, train_df, n=3)
    prompt = build_prompt(persona, target, context="general", few_shots=few_shots)
    
    output = call_llm(prompt, API_KEY)
    if output:
        rating, review = parse_response(output)
        pred_ratings.append(rating)
        true_ratings.append(true_rating)
        generated_reviews.append(review)
        true_reviews.append(true_review)
        print(f" Processed {i+1}/{len(eval_df)}")
    else:
        print(f" Failed at {idx}")
        time.sleep(2) 

 Processed 1/30
 Processed 2/30
 Processed 3/30
 Processed 4/30
 Processed 5/30
 Processed 6/30
 Processed 7/30
 Processed 8/30
 Processed 9/30
 Processed 10/30
 Processed 11/30
 Processed 12/30
 Processed 13/30
 Processed 14/30
 Processed 15/30
 Processed 16/30
 Processed 17/30
 Processed 18/30
 Processed 19/30
 Processed 20/30
 Processed 21/30
 Processed 22/30
 Processed 23/30
 Processed 24/30
 Processed 25/30
 Processed 26/30
 Processed 27/30
 Processed 28/30
 Processed 29/30
 Processed 30/30


In [7]:
# ===========================
# 4. Compute metrics
# ===========================
if pred_ratings:
    rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))
    mae = mean_absolute_error(true_ratings, pred_ratings)
    print(f"\nRMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = []
    for gen, ref in zip(generated_reviews, true_reviews):
        scores = scorer.score(ref, gen)
        rouge_scores.append(scores['rougeL'].fmeasure)

    avg_rouge = sum(rouge_scores) / len(rouge_scores)
    print(f"ROUGE-L F1: {avg_rouge:.4f}")



RMSE: 1.0488
MAE: 0.7000
ROUGE-L F1: 0.1245


In [9]:
# Compute ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for gen, ref in zip(generated_reviews, true_reviews):
    scores = scorer.score(ref, gen)          # returns dict: {'rouge1': Score, 'rouge2': Score, 'rougeL': Score}
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rouge2 = sum(rouge2_scores) / len(rouge2_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

print(f"ROUGE-1 F1: {avg_rouge1:.4f}")
print(f"ROUGE-2 F1: {avg_rouge2:.4f}")
print(f"ROUGE-L F1: {avg_rougeL:.4f}")

ROUGE-1 F1: 0.1805
ROUGE-2 F1: 0.0442
ROUGE-L F1: 0.1245
